In [1]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score
import json
import os

# CONFIG
TRAIN_CSV = r"C:\Users\ASUS\Downloads\CodeKubbb\Idekteppp\academy_alpha-i-week4\data\synthetic_plant_train.csv"
MODEL_OUT = r"C:\Users\ASUS\Downloads\CodeKubbb\Idekteppp\academy_alpha-i-week4\data\xgb_plant_model.json"
META_OUT  = r"C:\Users\ASUS\Downloads\CodeKubbb\Idekteppp\academy_alpha-i-week4\data\xgb_plant_model_meta.json"

FEATURES = ["temp_c", "humidity_pct", "lux", "vpd_kpa"]
TARGET = "y"
LABELS = [0, 1, 2]

#NOTE: Load TRAIN data only
train_df = pd.read_csv(TRAIN_CSV)
train_df = train_df.dropna(subset=FEATURES + [TARGET])

X_train = train_df[FEATURES].values
y_train = train_df[TARGET].astype(int).values

print("Train samples:", len(y_train))
print("Train class distribution:")
print(train_df[TARGET].value_counts())

#NOTE: Train model
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42,
)

model.fit(X_train, y_train)

train_pred = model.predict(X_train)
train_f1 = f1_score(y_train, train_pred, average="macro", labels=LABELS)
train_acc = accuracy_score(y_train, train_pred)

print(f"\nTRAIN Accuracy : {train_acc:.4f}")
print(f"TRAIN Macro-F1 : {train_f1:.4f}")

#NOTE: Save model + metadata
os.makedirs(os.path.dirname(MODEL_OUT), exist_ok=True)
model.save_model(MODEL_OUT)

meta = {
    "features": FEATURES,
    "labels": LABELS,
    "train_samples": int(len(y_train)),
    "train_macro_f1": float(train_f1),
    "train_accuracy": float(train_acc),
    "model_params": model.get_params(),
}

with open(META_OUT, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print(f"\n✅ Model saved to: {MODEL_OUT}")
print(f"🧾 Metadata saved to: {META_OUT}")

Train samples: 1958
Train class distribution:
y
1.0    1055
0.0     838
2.0      65
Name: count, dtype: int64

TRAIN Accuracy : 0.9990
TRAIN Macro-F1 : 0.9993

✅ Model saved to: C:\Users\ASUS\Downloads\CodeKubbb\Idekteppp\academy_alpha-i-week4\data\xgb_plant_model.json
🧾 Metadata saved to: C:\Users\ASUS\Downloads\CodeKubbb\Idekteppp\academy_alpha-i-week4\data\xgb_plant_model_meta.json
